# Scaffolded Spiral — matched-exposure training

Does progressive introduction, rehearsal, Binding specialization, and late rebalancing improve same-object Binding? Compare five fixed curricula using the **same 9,000 updates, exactly 45 presentations per question**, constant LR, and frozen Medium World model/data. This notebook trains; the separate analysis notebook reads its outputs.


In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path().cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SEEDS = [42, 43, 44, 45, 46]
RUN_SEEDS = SEEDS  # For a complete seed-42 pilot, use [42]; restore afterward.
BATCH_SIZE, EVAL_EVERY = 50, 100
LR, WEIGHT_DECAY, CLIP_NORM = 1e-4, 0.0, 1.0
MODEL_DIM, PATCH_SIZE, MODEL_DEPTH = 48, 4, 1
AUGMENT, DEVICE = True, "cuda"
DATA_DIR = PROJECT_ROOT / "outputs/medium_world/dataset"
OUT_DIR = PROJECT_ROOT / "outputs/medium_world/experiments/scaffolded_spiral"
RESULTS_FILE = OUT_DIR / "schedule_comparison.json"

In [2]:
import json, hashlib
from collections import Counter
import numpy as np
import pandas as pd
import torch
from torch import nn
from IPython.display import display
from src.medium_visual_world.learning import (
    TASKS, MediumDataset, AttributeBindingViT, augment_images, set_seed, environment
)
from src.medium_visual_world.scaffolded_spiral import SCHEDULES, make_spiral_blocks
from src.medium_visual_world.prediction_tracking import PredictionTrace, evaluate_predictions

device = torch.device(DEVICE)
torch.set_num_threads(min(4, torch.get_num_threads()))
train_data = MediumDataset(DATA_DIR / "train.jsonl")
id_dev = MediumDataset(DATA_DIR / "id_dev.jsonl")
ood_dev = MediumDataset(DATA_DIR / "ood_dev.jsonl")
hashes = {name: hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest()
          for name in ("train.jsonl", "id_dev.jsonl", "ood_dev.jsonl")}
TOTAL_STEPS = 9000
print(device, '|', len(train_data), 'training questions |', len(SCHEDULES), 'curricula')

cuda | 10000 training questions | 5 curricula


## Check the curriculum before training

The main schedule has **no Binding** during the mixed Color/Shape foundation or the Conjunction-with-rehearsal stage. It then specializes in Binding, rebalances the task mixture, and completes nine identical passes of common Mixed consolidation. Inspect the real schedules; run the separate audit script for exhaustive exposure verification.


In [ ]:
rows = []
for schedule in SCHEDULES:
    _, plan = make_spiral_blocks(train_data.rows, schedule, SEEDS[0])
    for phase in plan["phases"]:
        rows.append({"curriculum": schedule, "phase": phase["name"],
                     "start": phase["start_step"], "end": phase["end_step"],
                     **phase["by_template"]})
display(pd.DataFrame(rows))

,curriculum,phase,start,end,color,shape,conjunction,pair
0,scaffolded_spiral,attributes,0,2800,70000,70000,0,0
1,scaffolded_spiral,conjunction_rehearsal,2800,4450,7500,7500,67500,0
2,scaffolded_spiral,binding_specialization,4450,6100,5000,5000,7500,65000
3,scaffolded_spiral,rebalancing,6100,7200,7500,7500,15000,25000
4,scaffolded_spiral,common_mixed,7200,9000,22500,22500,22500,22500
5,spiral_late_focus,attributes,0,2800,70000,70000,0,0
6,spiral_late_focus,conjunction_rehearsal,2800,4450,7500,7500,67500,0
7,spiral_late_focus,early_rebalancing,4450,5550,7500,7500,15000,25000
8,spiral_late_focus,late_binding_focus,5550,7200,5000,5000,7500,65000
9,spiral_late_focus,common_mixed,7200,9000,22500,22500,22500,22500


## Train and save each completed run

Every seed starts from the same seed-specific model initialization. The active questions and their order vary by curriculum; the optimizer, augmentation and final exposure do not. Evaluate the same ID-dev and OOD-dev questions every 100 updates and at stage boundaries. The run summary and aligned prediction traces are saved separately after each **completed** run. The fixed-final checkpoint is the primary comparison.


In [ ]:
def binding_auc(history):
    pairs = [(h["step"], h["id_dev"]["by_template"]["pair"]) for h in history]
    return sum((b[0]-a[0])*(a[1]+b[1])/2 for a,b in zip(pairs[:-1], pairs[1:])) / TOTAL_STEPS

def task_shares(phase):
    return tuple(phase["by_template"][task] / phase["presentations"] for task in TASKS)

def run_one(schedule, seed):
    set_seed(seed)
    model = AttributeBindingViT(dim=MODEL_DIM, patch_size=PATCH_SIZE,
                                depth=MODEL_DEPTH).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    blocks, plan = make_spiral_blocks(train_data.rows, schedule, seed)
    boundary_steps = {p["end_step"] for p in plan["phases"][:-1]}
    eval_steps = set(range(0, TOTAL_STEPS+1, EVAL_EVERY)) | boundary_steps
    eval_steps |= {step+100 for step in boundary_steps}
    trace = PredictionTrace({"id_dev": id_dev, "ood_dev": ood_dev},
        metadata={"seed": seed, "schedule": schedule, "dataset_hashes": hashes})
    history, seen = [], Counter()
    step, running_loss, batches = 0, 0.0, 0

    def evaluate_step(phase):
        nonlocal running_loss, batches
        result = {}
        for split, data in (("id_dev", id_dev), ("ood_dev", ood_dev)):
            summary, predictions = evaluate_predictions(model, data, device, BATCH_SIZE)
            result[split] = summary
            trace.add(split, step, predictions)
        history.append({"step": step, "phase": phase, "lr": LR,
            "batch_train_loss": running_loss/batches if batches else None,
            "cumulative_task_presentations": dict(seen), **result})
        running_loss, batches = 0.0, 0
        print(f"{schedule:24} seed={seed} step={step:4} "
              f"ID Pair={result['id_dev']['by_template']['pair']:.3f} "
              f"OOD Pair={result['ood_dev']['by_template']['pair']:.3f}", flush=True)

    evaluate_step("initial")
    for phase, indices in blocks:
        for start in range(0, len(indices), BATCH_SIZE):
            sample_ids = indices[start:start+BATCH_SIZE]
            x, q, y = train_data.batch(sample_ids, device)
            if AUGMENT:
                x = augment_images(x)
            model.train()
            optimizer.zero_grad(set_to_none=True)
            loss = nn.functional.cross_entropy(model(x, q), y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
            optimizer.step()
            step += 1
            running_loss += loss.item()
            batches += 1
            seen.update(train_data.tasks[i] for i in sample_ids)
            if step in eval_steps:
                evaluate_step(phase)

    phases = plan["phases"]
    transitions = [{"step": a["end_step"], "from_phase": a["name"],
                    "to_phase": b["name"], "task_mix_changed": task_shares(a) != task_shares(b)}
                   for a,b in zip(phases[:-1], phases[1:])]
    trace_name = f"prediction_traces/{schedule}_seed_{seed}.npz"
    trace.save(OUT_DIR / trace_name)
    assert step == TOTAL_STEPS
    return {"seed": seed, "schedule": schedule, "status": "completed",
            "steps_run": step, "phase_plan": plan, "transitions": transitions,
            "history": history, "final": history[-1],
            "id_pair_aulc": binding_auc(history), "prediction_trace": trace_name}

In [ ]:
CONFIG = {"schedules": list(SCHEDULES), "seeds": SEEDS, "batch_size": BATCH_SIZE,
          "eval_every": EVAL_EVERY, "lr": LR, "weight_decay": WEIGHT_DECAY,
          "clip_norm": CLIP_NORM, "dim": MODEL_DIM, "patch_size": PATCH_SIZE,
          "depth": MODEL_DEPTH, "augmentation": AUGMENT,
          "total_steps": TOTAL_STEPS, "dataset_hashes": hashes}

def save_report(report):
    RESULTS_FILE.parent.mkdir(parents=True, exist_ok=True)
    temp = RESULTS_FILE.with_suffix(".json.tmp")
    temp.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    temp.replace(RESULTS_FILE)

report = (json.loads(RESULTS_FILE.read_text(encoding="utf-8")) if RESULTS_FILE.exists()
          else {"config": CONFIG, "environment": environment(), "runs": []})
assert report["config"] == CONFIG
completed = {(r["seed"], r["schedule"]) for r in report["runs"]}
for seed in RUN_SEEDS:
    for schedule in SCHEDULES:
        if (seed, schedule) in completed:
            print("Already completed:", seed, schedule)
            continue
        report["runs"].append(run_one(schedule, seed))
        save_report(report)
        print("Saved:", seed, schedule)
print("Completed", len(report["runs"]), "of", len(SEEDS)*len(SCHEDULES), "runs")

scaffolded_spiral        seed=42 step=   0 ID Pair=0.500 OOD Pair=0.500
scaffolded_spiral        seed=42 step= 100 ID Pair=0.492 OOD Pair=0.500
scaffolded_spiral        seed=42 step= 200 ID Pair=0.500 OOD Pair=0.500
scaffolded_spiral        seed=42 step= 300 ID Pair=0.480 OOD Pair=0.485
scaffolded_spiral        seed=42 step= 400 ID Pair=0.516 OOD Pair=0.492
scaffolded_spiral        seed=42 step= 500 ID Pair=0.500 OOD Pair=0.490
scaffolded_spiral        seed=42 step= 600 ID Pair=0.500 OOD Pair=0.485
scaffolded_spiral        seed=42 step= 700 ID Pair=0.504 OOD Pair=0.497
scaffolded_spiral        seed=42 step= 800 ID Pair=0.496 OOD Pair=0.482
scaffolded_spiral        seed=42 step= 900 ID Pair=0.500 OOD Pair=0.495
scaffolded_spiral        seed=42 step=1000 ID Pair=0.500 OOD Pair=0.500
scaffolded_spiral        seed=42 step=1100 ID Pair=0.500 OOD Pair=0.500
scaffolded_spiral        seed=42 step=1200 ID Pair=0.500 OOD Pair=0.497
scaffolded_spiral        seed=42 step=1300 ID Pair=0.500 OOD Pai

## Review saved runs

Partial runs are acceptable while piloting or executing seeds incrementally. **Do not interpret the final comparison until all five curricula have completed for all five seeds.** The analysis notebook requires the complete set.


In [ ]:
summary = pd.DataFrame([{"seed": r["seed"], "schedule": r["schedule"],
    "updates": r["steps_run"], "checkpoints": len(r["history"]),
    "ID_Binding": r["final"]["id_dev"]["by_template"]["pair"],
    "OOD_Binding": r["final"]["ood_dev"]["by_template"]["pair"]}
    for r in report["runs"]])
display(summary.sort_values(["seed", "schedule"]))

,seed,schedule,updates,checkpoints,ID_Binding,OOD_Binding
4,42,mixed,9000,91,0.656250,0.6125
3,42,prereq_mixed_pair_focus,9000,91,0.648438,0.6075
0,42,scaffolded_spiral,9000,93,0.640625,0.5050
1,42,spiral_late_focus,9000,95,0.636719,0.5225
2,42,staged_rehearsal,9000,93,0.640625,0.5500
9,43,mixed,9000,91,0.585938,0.5425
8,43,prereq_mixed_pair_focus,9000,91,0.605469,0.5350
5,43,scaffolded_spiral,9000,93,0.605469,0.5425
6,43,spiral_late_focus,9000,95,0.601562,0.5350
7,43,staged_rehearsal,9000,93,0.593750,0.5275
